In [1]:
#ISM 4641 Project-Ha Tran and Adithya Venkatramanan

In [2]:
#Import libraries:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
#Data Inspection
df=pd.read_csv(r"C:\Users\svrad\OneDrive\University of South Florida\Semester 5\ISM 4641\Project Content\Electronic_sales_Sep2023-Sep2024.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\svrad\\OneDrive\\University of South Florida\\Semester 5\\ISM 4641\\Project Content\\Electronic_sales_Sep2023-Sep2024.csv'

In [ ]:
df.head(5)

In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
#Missing Data
df.isnull().sum()*100/len(df)

In [ ]:
df['Add-ons Purchased'] = df['Add-ons Purchased'].fillna('None')
df

In [ ]:
numeric_df = df.select_dtypes(include=['number'])

In [ ]:
# Calculate the counts for 'M' and 'F'
gender_counts = df['Gender'].value_counts()
total = gender_counts.sum()

# Calculate probabilities for each gender
prob_male = gender_counts['Male'] / total
prob_female = gender_counts['Female'] / total

# Define a function to impute missing values
def impute_gender(x):
    if pd.isna(x) or x == ' ':
        return np.random.choice(['Male', 'Female'], p=[prob_male, prob_female])
    return x

# Apply the function to fill missing values in the Gender column
df['Gender'] = df['Gender'].apply(impute_gender)

In [ ]:
df.isnull().sum()*100/len(df)

In [ ]:
df['Final Price']=df['Total Price']+df['Add-on Total']
df

In [ ]:
#Summary Statistics
df.describe()

In [ ]:
# Correlation
corr = df.corr(numeric_only=True)

# Set up the matplotlib figure
f, ax = plt.subplots(figsize=(11, 9))

# Generate a custom diverging colormap
cmap = sns.diverging_palette(230, 20, as_cmap=True)

# Draw the heatmap with the correct aspect ratio
sns.heatmap(corr, cmap=cmap, vmax=1, center=0, square=True, linewidths=.5, cbar_kws={"shrink": .5})
plt.show()

In [ ]:
# Plots
# Create age groups
bins = [0,34,64,100]  # Define age ranges
labels = ['0-34','35-64','65+']
df['Age Group'] = pd.cut(df['Age'], bins=bins, labels=labels, right=False)
# Group data by Age and Gender and sum the profits
grouped_data = df.groupby(['Age Group', 'Gender'])['Final Price'].sum().unstack()
ax=grouped_data.plot(kind='bar', stacked=True, figsize=(12, 6), colormap='Spectral')
# Customize the chart
plt.title('Revenue by Age Group and Gender', fontsize=14)
# Add annotations
for i, (age_group, gender_data) in enumerate(grouped_data.iterrows()):
    cumulative = 0  # Initialize the cumulative value to track the position of each gender in the stack
    for j, gender in enumerate(grouped_data.columns):
        value = gender_data[gender]
        if value > 0:
            # Position the annotation at the top of each stacked segment
            ax.text(
                i,  # x-coordinate (age group index)
                cumulative + value / 2,  # y-coordinate (half of the segment height)
                f'${value:,.2f}',  # Format the value as currency
                ha='center',  # Horizontal alignment
                va='center',  # Vertical alignment (center within the segment)
                fontsize=10,  # Font size
                color='white'  # Set annotation color to red
            )
            cumulative += value  # Update cumulative value for next segment
plt.xlabel('Age', fontsize=12)
plt.ylabel('Revenue ($)', fontsize=12)
plt.legend(title='Gender', fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

# Show the chart
plt.show()

In [ ]:
Revenue = df.groupby('Product Type')['Total Price'].sum()
for index, value in enumerate(Revenue.values):
    plt.text(
        index,                   # x-coordinate of the text
        value ,  # y-coordinate slightly above the bar
        f'${value:,.2f}',        # Format value as currency (e.g., $1,234.56)
        ha='center',             # Horizontal alignment
        va='bottom',             # Vertical alignment
        fontsize=10              # Font size for annotations
    )
plt.bar(Revenue.index, Revenue.values, color = "#F8275B")
plt.xlabel('Product Type')
plt.ylabel('Revenue ($)')
plt.title('Sales Comparison Between Product Types')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
#ADD CODE FOR LATEST VISUAL

In [ ]:
# Count the occurrences of each unique rating
rating_counts = df['Rating'].value_counts().sort_index()

# Create a pie chart
plt.pie(rating_counts, labels=rating_counts.index, autopct='%1.1f%%', colors=plt.cm.Paired.colors)
plt.title('Rating Distribution')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Ensure 'Purchase Date' is in datetime format
df['Purchase Date'] = pd.to_datetime(df['Purchase Date'], errors='coerce')

# Extract the day of the week from 'Purchase Date'
df['Day of Week'] = df['Purchase Date'].dt.day_name()

# Group by day of the week and calculate the total 'Quantity'
total_quantity_by_day = df.groupby('Day of Week')['Quantity'].sum()

# Sort days of the week in chronological order
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
total_quantity_by_day = total_quantity_by_day.reindex(day_order)

# Plotting
plt.figure(figsize=(10, 6))
total_quantity_by_day.plot(kind='line', marker='o', color='skyblue', linestyle='-', linewidth=2, markersize=8, markeredgecolor='black')
plt.xlabel('Day of the Week')
plt.ylabel('Total Quantity')
plt.title('Total Quantity by Day of the Week')
plt.xticks(rotation=45)  # Rotate x-axis labels for readability
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
# Ensure 'Purchase Date' is in datetime format
df['Purchase Date'] = pd.to_datetime(df['Purchase Date'], errors='coerce')

# Extract the year and month from 'Purchase Date' in a combined format (e.g., "2023-September")
df['Year-Month'] = df['Purchase Date'].dt.strftime('%Y-%B')

# Group by 'Product Type' and 'Year-Month' to calculate the total 'Quantity'
total_quantity_by_product_month = df.groupby(['Product Type', 'Year-Month'])['Quantity'].sum().unstack()

# Sort columns (Year-Month) chronologically
total_quantity_by_product_month = total_quantity_by_product_month[
    sorted(total_quantity_by_product_month.columns, key=lambda x: pd.to_datetime(x, format='%Y-%B'))
]

# Plotting
plt.figure(figsize=(14, 8))
for product in total_quantity_by_product_month.index:
    plt.plot(
        total_quantity_by_product_month.columns,
        total_quantity_by_product_month.loc[product],
        marker='o',
        linewidth=2,
        label=product
    )

# Add labels and title
plt.xlabel('Month', fontsize=12)
plt.ylabel('Total Quantity Sold', fontsize=12)
plt.title('Total Quantity Sold for Each Product Type by Month', fontsize=14)
plt.xticks(rotation=45, ha='right')  # Rotate x-axis labels for readability
plt.legend(title='Product Type', bbox_to_anchor=(1.05, 1), loc='upper left')  # Adjust legend position
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

# Show the plot
plt.show()


In [ ]:
df.head()

In [ ]:
# Non-numeric feature
from sklearn.preprocessing import OneHotEncoder

# Initialize OneHotEncoder
encoder = OneHotEncoder(drop=None, sparse_output=False)

# Fit and transform the Gender column
encoded = encoder.fit_transform(df[['Gender']])

# Create a DataFrame with the encoded columns
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(['Gender']), index=df.index)

# Merge the encoded columns back to the original DataFrame
df = pd.concat([df, encoded_df], axis=1)

In [ ]:
encoder = OneHotEncoder(drop=None, sparse_output=False)

# Fit and transform the Gender column
encoded = encoder.fit_transform(df[['Loyalty Member']])

# Create a DataFrame with the encoded columns
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(['Loyalty Member']), index=df.index)

# Merge the encoded columns back to the original DataFrame
df = pd.concat([df, encoded_df], axis=1)

In [ ]:
encoder = OneHotEncoder(drop=None, sparse_output=False)

# Fit and transform the Gender column
encoded = encoder.fit_transform(df[['Order Status']])

# Create a DataFrame with the encoded columns
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(['Order Status']), index=df.index)

# Merge the encoded columns back to the original DataFrame
df = pd.concat([df, encoded_df], axis=1)

In [ ]:
encoder = OneHotEncoder(drop=None, sparse_output=False)

# Fit and transform the Payment Method column
encoded = encoder.fit_transform(df[['Payment Method']])

# Create a DataFrame with the encoded columns
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(['Payment Method']), index=df.index)

# Merge the encoded columns back to the original DataFrame
df = pd.concat([df, encoded_df], axis=1)

In [ ]:
encoder = OneHotEncoder(drop=None, sparse_output=False)

# Fit and transform the Payment Method column
encoded = encoder.fit_transform(df[['Product Type']])

# Create a DataFrame with the encoded columns
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(['Product Type']), index=df.index)

# Merge the encoded columns back to the original DataFrame
df = pd.concat([df, encoded_df], axis=1)

In [ ]:
encoder = OneHotEncoder(drop=None, sparse_output=False)

# Fit and transform the Payment Method column
encoded = encoder.fit_transform(df[['Shipping Type']])

# Create a DataFrame with the encoded columns
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(['Shipping Type']), index=df.index)

# Merge the encoded columns back to the original DataFrame
df = pd.concat([df, encoded_df], axis=1)

In [ ]:
df=df.drop('Gender', axis=1)
df=df.drop('Loyalty Member', axis=1)
df=df.drop('Product Type', axis=1)
df=df.drop('Order Status', axis=1)
df=df.drop('Payment Method', axis=1)
df=df.drop('Shipping Type', axis=1)
df=df.drop('Day of Week', axis=1)

df.head()

In [ ]:
#Feature Creation

from sklearn.preprocessing import StandardScaler
df['Purchase Day'] = pd.to_datetime(df['Purchase Date']).dt.day_name()  # Day of the week
df['Purchase Month'] = pd.to_datetime(df['Purchase Date']).dt.month  # Month

In [ ]:
df.drop(['Purchase Date'], axis=1, inplace=True)

In [ ]:
df = pd.get_dummies(df, columns=['Purchase Day'], drop_first=False)

In [ ]:
#Pandas is storing days of week as boolean instead of binary to optimize performance so we will be calculating accuracy scores instead of MAE

In [ ]:
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(df, test_size=0.2, random_state=42)

In [ ]:
# Split train and test sets
# For training data
train_X = train_set.drop(['Customer ID', 'SKU', 'Add-ons Purchased', 'Age Group', 'Shipping Type_Overnight'], axis=1)  # drop labels for training set
train_y = train_set['Shipping Type_Overnight'].copy()

# For test data
test_X = test_set.drop(['Customer ID', 'SKU', 'Add-ons Purchased', 'Age Group', 'Shipping Type_Overnight'], axis=1)  # drop labels for training set
test_y = test_set['Shipping Type_Overnight'].copy()

In [ ]:
train_X

In [ ]:
test_X

In [ ]:
# Model training
from sklearn.svm import SVC
svc_model = SVC()
svc_model.fit(train_X, train_y)

In [ ]:
#Explain, predict, or both
yhat_train = svc_model.predict(train_X)
yhat_test = svc_model.predict(test_X)

In [ ]:
# Evaluation
from sklearn.metrics import mean_absolute_error

In [ ]:
#evaluate in-sample predictions
mean_absolute_error(train_y, yhat_train)

In [ ]:
#evaluate out-of-sample predictions
mean_absolute_error(test_y, yhat_test)

In [ ]:
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(df, test_size=0.2, random_state=42)

In [ ]:
# Split train and test sets
# For training data
train_X = train_set.drop(['Customer ID', 'SKU', 'Add-ons Purchased', 'Age Group', 'Loyalty Member_Yes'], axis=1)  # drop labels for training set
train_y = train_set['Loyalty Member_Yes'].copy()

# For test data
test_X = test_set.drop(['Customer ID', 'SKU', 'Add-ons Purchased', 'Age Group', 'Loyalty Member_Yes'], axis=1)  # drop labels for training set
test_y = test_set['Loyalty Member_Yes'].copy()

In [ ]:
# Model training
svc_model_2 = SVC()
svc_model_2.fit(train_X, train_y)

In [ ]:
#Explain, predict, or both
yhat_train = svc_model_2.predict(train_X)
yhat_test = svc_model_2.predict(test_X)

In [ ]:
# Evaluation
from sklearn.metrics import mean_absolute_error

In [ ]:
#evaluate in-sample predictions
mean_absolute_error(train_y, yhat_train)

In [ ]:
#evaluate out-of-sample predictions
mean_absolute_error(test_y, yhat_test)

In [ ]:
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(df, test_size=0.2, random_state=42)

In [ ]:
# Split train and test sets
# For training data
train_X = train_set.drop(['Customer ID', 'SKU', 'Add-ons Purchased', 'Age Group', 'Purchase Day_Thursday'], axis=1)  # drop labels for training set
train_y = train_set['Purchase Day_Thursday'].copy()

# For test data
test_X = test_set.drop(['Customer ID', 'SKU', 'Add-ons Purchased', 'Age Group', 'Purchase Day_Thursday'], axis=1)  # drop labels for training set
test_y = test_set['Purchase Day_Thursday'].copy()

In [ ]:
# Model training
svc_model_3 = SVC()
svc_model_3.fit(train_X, train_y)

In [ ]:
#Explain, predict, or both
yhat_train = svc_model_3.predict(train_X)
yhat_test = svc_model_3.predict(test_X)

In [ ]:
# Evaluation
from sklearn.metrics import accuracy_score

In [ ]:
#evaluate in-sample predictions
print(accuracy_score(train_y, yhat_train))

In [ ]:
#evaluate out-of-sample predictions
print(accuracy_score(test_y, yhat_test))